# **ginger**

In [1]:
import sys
import os

if "google.colab" in sys.modules:
    import getpass

    token = getpass.getpass()

    repo_url = f"https://{token}@github.com/Mango-Cats/ginger.git"

    if os.path.exists("ginger"):
        %cd ginger
        !git pull origin main
    else:
        !git clone {repo_url}
        %cd ginger

    try:
        %load_ext autoreload
        %autoreload 2
    except Exception as e:
        print("autoreload not available:", e)

    !pip install -e . --no-deps

/content/ginger
From https://github.com/Mango-Cats/ginger
 * branch            main       -> FETCH_HEAD
Already up to date.
autoreload not available: No module named 'imp'
Obtaining file:///content/ginger
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for ginger (pyproject.toml) ... done
  Created wheel for ginger: filename=ginger-0.1.0-0.editable-py3-none-any.whl size=5480 sha256=7da328bce176a4e72fbe964d99a844c23f9e37b18be6e0038a1cc90de7ab6f7c
  Stored in directory: /tmp/pip-ephem-wheel-cache-a4t6i57u/wheels/4a/45/44/d3f200324dc86253cc07b97b9015a378520aae0b30f78eb83b
Successfully built ginger
  Attempting uninstall: ginger
    Found existing installation: ginger 0.1.0
    Uninstalling ginger-0.1.0:
      Successfully uninstalled ginger-0.1.0


In [2]:
import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(torch.__version__)
print(DEVICE)

2.10.0+cu128
cuda


## **Data Loading & Preprocessing**

In [3]:
from src.dataman import create_dataloaders
from src.dataman.constants import PAD_IDX
from pathlib import Path

In [4]:
DATA_PATH = Path("data/wikipron_tagalog.tsv")
BATCH_SIZE = 64

train_loader, val_loader, test_loader, grapheme_vocab, phoneme_vocab = create_dataloaders(
    DATA_PATH,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

print(f"Train samples: {len(train_loader.dataset)}")
print(f"Val samples: {len(val_loader.dataset)}")
print(f"Test samples: {len(test_loader.dataset)}")
print(f"Grapheme vocab size: {len(grapheme_vocab)}")
print(f"Phoneme vocab size: {len(phoneme_vocab)}")

Train samples: 14590
Val samples: 1824
Test samples: 1824
Grapheme vocab size: 32
Phoneme vocab size: 34


## **Model Initialization**

In [5]:
from src.architecture.transformer import Transformer
from torch.optim.lr_scheduler import OneCycleLR
from torch.optim import AdamW
import torch.nn as nn

In [6]:
EMBEDDING_DIM = 128
NUM_ATTENTION_HEADS = 4
NUM_ENCODER_LAYERS = 4
NUM_DECODER_LAYERS = 4
FEEDFORWARD_DIM = 512
DROPOUT_RATE = 0.1
LEARNING_RATE = 0.001
WEIGHT_DECAY = 1e-4
NUM_EPOCHS = 100 

model = Transformer(
    num_graphemes=len(grapheme_vocab),
    num_phonemes=len(phoneme_vocab),
    embedding_dim=EMBEDDING_DIM,
    num_attention_heads=NUM_ATTENTION_HEADS,
    num_encoder_layers=NUM_ENCODER_LAYERS,
    num_decoder_layers=NUM_DECODER_LAYERS,
    feedforward_dim=FEEDFORWARD_DIM,
    dropout_rate=DROPOUT_RATE,
).to(DEVICE)

model = torch.compile(model)

CRITERION = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
OPTIMIZER = AdamW(params=model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
SCHEDULER = OneCycleLR(
    optimizer=OPTIMIZER,
    max_lr=LEARNING_RATE,
    steps_per_epoch=len(train_loader),
    epochs=NUM_EPOCHS,
    pct_start=0.1 
)
SCALER = torch.cuda.amp.GradScaler()
PATIENCE = 7

/tmp/ipykernel_25054/2010783063.py:33: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  SCALER = torch.cuda.amp.GradScaler()


## **Training**

In [7]:
from src.trainer import train_model

In [8]:
results = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=CRITERION,
    optimizer=OPTIMIZER,
    scheduler=SCHEDULER,
    scaler=SCALER,
    device=DEVICE,
    num_epochs=NUM_EPOCHS,
    patience=PATIENCE,
)

E0501 13:36:06.418000 25054 torch/_subclasses/fake_tensor.py:2827] [0/0] failed while attempting to run meta for aten.add.Tensor
E0501 13:36:06.418000 25054 torch/_subclasses/fake_tensor.py:2827] [0/0] Traceback (most recent call last):
E0501 13:36:06.418000 25054 torch/_subclasses/fake_tensor.py:2827] [0/0]   File "/usr/local/lib/python3.12/dist-packages/torch/_subclasses/fake_tensor.py", line 2823, in _dispatch_impl
E0501 13:36:06.418000 25054 torch/_subclasses/fake_tensor.py:2827] [0/0]     r = func(*args, **kwargs)
E0501 13:36:06.418000 25054 torch/_subclasses/fake_tensor.py:2827] [0/0]         ^^^^^^^^^^^^^^^^^^^^^
E0501 13:36:06.418000 25054 torch/_subclasses/fake_tensor.py:2827] [0/0]   File "/usr/local/lib/python3.12/dist-packages/torch/_ops.py", line 819, in __call__
E0501 13:36:06.418000 25054 torch/_subclasses/fake_tensor.py:2827] [0/0]     return self._op(*args, **kwargs)
E0501 13:36:06.418000 25054 torch/_subclasses/fake_tensor.py:2827] [0/0]            ^^^^^^^^^^^^^^^^^^^


Starting training for 100 epochs...



TorchRuntimeError: Dynamo failed to run FX node with fake tensors: call_function <built-in function add>(*(FakeTensor(..., device='cuda:0', size=(19, 64, 128), grad_fn=<MulBackward0>), FakeTensor(..., device='cuda:0', size=(1, 5000, 128))), **{}): got RuntimeError('Attempting to broadcast a dimension of length 5000 at -2! Mismatching argument at index 1 had torch.Size([1, 5000, 128]); but expected shape should be broadcastable to [19, 64, 128]')

from user code:
   File "/content/ginger/src/architecture/transformer.py", line 66, in forward
    src = self.positional_encoder(src)
  File "/content/ginger/src/architecture/positional_encoder.py", line 47, in forward
    embeddings = embeddings + self.pe[:seq_len, :]

Set TORCHDYNAMO_VERBOSE=1 for the internal stack trace (please do this especially if you're reporting a bug to PyTorch). For even more developer context, set TORCH_LOGS="+dynamo"


## **Inference**

In [ ]:
from src.evaluation.evaluate_model import evaluate_model

ModuleNotFoundError: No module named 'src.evaluation.evaluate_model'

In [ ]:
eval_results = evaluate_model(
    model=model,
    test_loader=test_loader,
    grapheme_vocab=grapheme_vocab,
    phoneme_vocab=phoneme_vocab,
    device=DEVICE,
)